In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/shopee-product-matching/sample_submission.csv
/kaggle/input/competitions/shopee-product-matching/train.csv
/kaggle/input/competitions/shopee-product-matching/test.csv


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.preprocessing
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix,mean_squared_error

In [ ]:
# Part A
# Q1

# Create a new code cell and run this:
train_df = pd.read_csv('/kaggle/input/competitions/shopee-product-matching/train.csv')

# View the first 5 rows of the dataset
train_df.head()
train_df.columns
""" posting id represents the unique producct for the speciific product listing on shopee
image reprsents the specific image number(or code) in filename format for that particular image of the images set
image_phash represents the image perceptual hash(phash) of the image encoding the visual content of the image. Imgaes very similar in appearance will have similar p-hashes making it eaier to reference duplicate images
title represents the product text title/description
label group-The "ground truth" target variable. All postings that represent the exact same actual product share the same label_group(Unique product label)"""

""" A product group is represented by label_group column which is to be predicted for test data based"""
"""The matching relationship is represented by the label_group column. If two different posting_ids share the exact same label_group number, they are a match. You can think of the label_group as the true product ID.
When the final model is build]t and submitted the predictions, we don't use label_group anymore. Instead, the matching relationship is represented by a column called matches. For every posting_id, the model must output a space-separated string of all the other posting_ids that you believe are the same product."""

In [ ]:
#Part A
#Q2

train_df['posting_id'].value_counts #34250

In [ ]:
train_df['label_group'].nunique()  #11014 unique label groups

In [ ]:
group_sizes= train_df.groupby('label_group').size()
print("Group size counts")
print(group_sizes.value_counts())

plt.figure(figsize=(10, 5))
sns.histplot(group_sizes, bins=50, kde=False)
plt.title('Distribution of Product Group Sizes')
plt.xlabel('Number of Listings in a Group')
plt.ylabel('Frequency (Number of Groups)')
plt.show()

In [ ]:
num_unique= train_df['image'].nunique()
num_duplicated = train_df['image'].size - num_unique
print(num_duplicated)
print(num_unique)
#Duplicate images with ids

duplicates_df = train_df[train_df.duplicated(subset=['image'], keep=False)]


duplicates_df = duplicates_df.sort_values(by='image')

# Print the posting_id and the image name to see the duplicates
print(duplicates_df[['posting_id', 'image']])

In [ ]:
# Find all rows with exact duplicate titles
duplicate_titles_df = train_df[train_df.duplicated(subset=['title'], keep=False)]

# Sort by title so duplicates appear right next to each other
duplicate_titles_df = duplicate_titles_df.sort_values(by='title')

# View the total count and the first few rows
print(f"Total listings with an exact duplicate title: {len(duplicate_titles_df)}")
print(duplicate_titles_df[['posting_id', 'title']].head(10))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
tfidf=TfidfVectorizer(stop_words='english',max_features=10000)
tfidf_matrix=tfidf.fit_transform(train_df['title'])
target_index=0
target_title=train_df.iloc[target_index]['title']
similarity_scores= cosine_similarity(tfidf_matrix[target_index], tfidf_matrix).flatten()
results_df=pd.DataFrame(
    {
        'posting_id' : train_df['posting_id'],
        'title': train_df['title'],
        'similarity_scores': similarity_scores
    }
)
top_matches=results_df.sort_values(by='similarity_scores', ascending=False).head(5)
print(top_matches)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. Initialize and fit the TF-IDF vectorizer
tfidf = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = tfidf.fit_transform(train_df['title'])

print("Calculating all-vs-all similarity matrix... (This might take a few seconds)")

# 2. Calculate the massive similarity grid for ALL items at once
# This returns a 34250 x 34250 grid of scores
all_similarities = cosine_similarity(tfidf_matrix, tfidf_matrix)

# 3. Set a similarity threshold to decide what counts as a "match"
# 1.0 is exact match, 0.0 is completely different. 0.7 is a good starting baseline.
THRESHOLD = 0.70 
all_matches = []

# 4. Loop through the grid to extract matches for every single item
for i in range(len(train_df)):
    # Find the row indexes where the similarity score is above our threshold
    match_indexes = np.where(all_similarities[i] > THRESHOLD)[0]
    
    # Get the actual 'posting_id's for those matched rows
    matched_posting_ids = train_df.iloc[match_indexes]['posting_id'].values
    
    # Join them into a space-separated string (This is the Kaggle submission format!)
    matches_string = ' '.join(matched_posting_ids)
    all_matches.append(matches_string)

# 5. Add our predicted matches as a new column in our dataset
train_df['predicted_title_matches'] = all_matches

# View the results!
print(train_df[['posting_id', 'title', 'predicted_title_matches']].head(10))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


train_df['title_word_count'] = train_df['title'].apply(lambda x: len(str(x).split()))

# Plot the distribution
plt.figure(figsize=(10, 5))
sns.histplot(train_df['title_word_count'], bins=35, kde=True, color='teal')
plt.title('Distribution of Title Word Counts')
plt.xlabel('Number of Words in Title')
plt.ylabel('Number of Listings')
plt.show()

print(f"Average title length: {train_df['title_word_count'].mean():.2f} words")
print(f"Maximum title length: {train_df['title_word_count'].max()} words")

In [ ]:
from collections import Counter
import pandas as pd

all_words = ' '.join(train_df['title'].astype(str).str.lower()).split()

word_counts = Counter(all_words)
top_20_words = pd.DataFrame(word_counts.most_common(20), columns=['Word', 'Count'])

plt.figure(figsize=(10, 6))
sns.barplot(data=top_20_words, x='Count', y='Word', palette='viridis')
plt.title('Top 20 Most Frequent Words in Product Titles')
plt.xlabel('Frequency')
plt.ylabel('Word')
plt.show()

In [ ]:
total_listings = len(train_df)
unique_images = train_df['image'].nunique()
reused_images = total_listings - unique_images

plt.figure(figsize=(6, 6))
plt.pie([unique_images, reused_images], 
        labels=['Unique Images', 'Reused/Duplicate Images'], 
        autopct='%1.1f%%', 
        colors=['#66b3ff', '#ff9999'], 
        startangle=90)
plt.title('Proportion of Unique vs Reused Images')
plt.show()

In [ ]:
#PART A
#Q3
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# Base path for the images in this Kaggle dataset
IMAGE_DIR = '/kaggle/input/competitions/shopee-product-matching/train_images/'

def plot_listings(df_subset, title_text):
    """Helper function to plot a row of product images and their titles"""
    fig, axes = plt.subplots(1, len(df_subset), figsize=(15, 5))
    fig.suptitle(title_text, fontsize=16, fontweight='bold')
    
    # Handle single image case
    if len(df_subset) == 1: axes = [axes]
        
    for ax, (_, row) in zip(axes, df_subset.iterrows()):
        img_path = os.path.join(IMAGE_DIR, row['image'])
        try:
            img = mpimg.imread(img_path)
            ax.imshow(img)
        except FileNotFoundError:
            ax.text(0.5, 0.5, 'Image Not Found', ha='center')
            
        ax.axis('off')
        # Truncate title so it fits on the plot
        short_title = row['title'][:40] + '...' if len(row['title']) > 40 else row['title']
        ax.set_title(short_title, fontsize=10, wrap=True)
    plt.tight_layout()
    plt.show()

In [ ]:
# Find a label_group with exactly 4 listings
common_group_id = train_df['label_group'].value_counts()[train_df['label_group'].value_counts() == 4].index[0]
same_products = train_df[train_df['label_group'] == common_group_id]

plot_listings(same_products, f"Same Product (Label Group: {common_group_id})")
"""Why it's difficult: Sellers might use different lighting, add promotional text/watermarks to the image, or translate the title into different languages. Your model needs to look past the clutter to see the core item."""

In [ ]:
# Group by label and count unique image hashes
phash_counts = train_df.groupby('label_group')['image_phash'].nunique()
# Find a group where the same product has at least 3 visually distinct images
diff_img_group_id = phash_counts[phash_counts >= 3].index[0]
diff_images_same_prod = train_df[train_df['label_group'] == diff_img_group_id].drop_duplicates(subset=['image_phash']).head(3)

plot_listings(diff_images_same_prod, "Same Product, Different Images")
"""Why it's difficult: Computer vision models (like CNNs) will struggle here because the pixels are entirely different (e.g., front of the box vs. back of the box, or a lifestyle photo vs. a plain white background). The model has to rely heavily on the text titles to realize they are the same thing."""

In [ ]:
# Let's search for the word "iphone" and see how it tricks basic matching
iphone_listings = train_df[train_df['title'].str.lower().str.contains('iphone 12')].head(10)
# Pick two that have different label groups
similar_titles_diff_prods = iphone_listings.drop_duplicates(subset=['label_group']).head(3)

plot_listings(similar_titles_diff_prods, "Similar Titles, Different Products")
"""Why it's difficult: Sellers stuff titles with keywords. "iPhone 12 Case" and "iPhone 12 Screen Protector" share most of their text, but are entirely different products. NLP models might falsely group them together if they don't pay attention to the specific modifier words."""

In [ ]:
# Group by phash and count unique label_groups
group_counts_per_phash = train_df.groupby('image_phash')['label_group'].nunique()
# Find an image that was used for completely different products
deceptive_phash = group_counts_per_phash[group_counts_per_phash > 1].index[0]
same_img_diff_prods = train_df[train_df['image_phash'] == deceptive_phash].drop_duplicates(subset=['label_group']).head(3)

plot_listings(same_img_diff_prods, "Same Image, Different Products")
"""Why it's difficult: This is a nightmare for image-matching models! Sometimes sellers use a generic brand poster, a multi-item collage, or the exact same shoe photo but one listing is for "Size 8" and the other is for "Size 10". If the physical product variation (like size or memory capacity) isn't in the picture, the text is the only saving grace."""

In [ ]:
Listings with noisy or incomplete information
Let's look at listings where the seller provided almost zero effort—for example, titles with only 1 or 2 words.

Python
# Find titles with less than 3 words using the column we created earlier
noisy_listings = train_df[train_df['title_word_count'] <= 2].head(4)

plot_listings(noisy_listings, "Noisy / Incomplete Information")
"""Why it's difficult: If a seller just lists the product as "Shoes" or uses random emoji spam, there is no useful text data to compare against other listings. The model has to rely 100% on the image to find matches."""

## Part 2: Identified Challenges & Observations
Based on the exploratory data analysis and visual investigations above, I have identified several major challenges a machine learning model will face in this dataset:

**1. Deceptive Stock Photos (Same Image, Different Product):** 
Many sellers use the exact same manufacturer stock photo (identical `image_phash`) to sell different variations of a product (e.g., a phone case for an iPhone 12 vs. an iPhone 12 Pro, or a size 8 shoe vs. a size 10). A model relying only on images will completely fail here.

**2. Extreme Visual Variance (Different Image, Same Product):** 
Sellers photograph the same item in wildly different ways. One might use a clean white background, another might take a low-lighting picture on their bed, and another might just upload a picture of the cardboard box. Computer vision models will struggle to link these without text context.

**3. Keyword Stuffing (Similar Title, Different Product):** 
To show up in search results, sellers stuff titles with keywords. "iPhone 12 case" and "iPhone 12 screen protector" will have extremely high TF-IDF similarity scores because they share most of their text, leading to false-positive matches.

**4. Noisy and Incomplete Titles:** 
Some listings have only 1 or 2 words (e.g., just "shoes" or random emojis). With no text data to rely on, the model is forced to make a guess based entirely on the image.

**5. Image Watermarks and Text:** 
Shopee sellers frequently edit their images to add giant promotional text, prices, or borders (e.g., "50% OFF!"). This obscures the actual product pixels, making it harder for image models to extract the core item features.

---

## Part 3: Conclusions (Questions to Consider)

* **What makes two listings belong to the same product?**
Two listings belong to the same product (share the same `label_group`) if they represent the exact same physical item, regardless of how the seller chose to photograph, name, or price it. 

* **Can product titles alone reliably determine whether two listings match?**
No. While TF-IDF and NLP are strong baseline tools, relying solely on text fails when sellers use keyword stuffing for accessories (false positives) or when they provide lazy, 1-word descriptions (false negatives).

* **Can images alone reliably determine whether two listings match?**
No. Relying solely on perceptual hashes or CNNs fails when sellers use the exact same generic stock photo for different sizes/models, or when the exact same product is photographed in completely different lighting conditions.

* **What types of examples are likely to be difficult for a machine learning model?**
The most difficult examples will be "Accessory vs. Main Product" listings (where the image and title look like a phone, but it's actually just a phone case) and "Combo/Bundle" listings where multiple items are featured in the same picture.

* **What information in the dataset could potentially be useful for solving the problem?**
To build a highly accurate model, we must **combine** both modalities. We can use NLP (like TF-IDF or BERT) on the `title` column, combine it with a Convolutional Neural Network (CNN) feature extractor on the `image` column, and use the `image_phash` to quickly group identical photos.